# Run Joni's project in Google Colab
Upload the delivered **Joni_PSD_UTS.zip**. Keep this tab connected while jobs run. After training, download and save the new results ZIP. Remote processing and credentials are required; no model results are simulated.

In [ ]:
from google.colab import files
import zipfile, pathlib
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith(".zip"))
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall("/content/joni_project")
root = pathlib.Path("/content/joni_project/Joni_PSD_UTS")
assert (root / "uts/code/crawling.py").exists()

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"-m","pip","install","-r",str(root/"requirements.txt")],check=True)

In [ ]:
subprocess.run([sys.executable,str(root/"uts/code/crawling.py"),"--plan"],check=True)

## Authenticate and process
Open the device login link printed below and sign in to Copernicus. Processing can take a long time. Check credits before running. If Colab disconnects before files are downloaded, remote jobs may still be active; review the Copernicus dashboard before resubmitting.

In [ ]:
subprocess.run([sys.executable,str(root/"uts/code/crawling.py")],check=True)

In [ ]:
subprocess.run([sys.executable,str(root/"uts/code/pipeline.py"),"train"],check=True)

In [ ]:
import pandas as pd
display(pd.read_csv(root/"uts/deploy/ringkasan_model_knn.csv"))

## Download your real results
Raw downloaded GeoTIFFs and token/secret files are excluded. Extract the ZIP and upload the project contents to GitHub.

In [ ]:
output="/content/Joni_PSD_UTS_results.zip"
with zipfile.ZipFile(output,"w",zipfile.ZIP_DEFLATED) as z:
    for path in root.rglob("*"):
        if not path.is_file(): continue
        if path.suffix.lower() in {".tif",".tiff",".pyc"}: continue
        if any(p.startswith("job_") or p in {"__pycache__",".venv"} for p in path.parts): continue
        if path.name in {"secrets.toml","daftar_job.csv"}: continue
        z.write(path,"Joni_PSD_UTS/"+str(path.relative_to(root)))
files.download(output)